In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import argparse
import scanpy as sc
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import sys
sys.path.append("..")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
import anndata as ad
import pandas as pd
from collections import Counter
import anndata
from sklearn.model_selection import train_test_split
from lib_vqvae.vqvae import VQVAE
from lib_vqvae.dataset import *

In [ ]:
# Load preprocessed AnnData for trajectory analysis
import scanpy as sc
import os

# Get project root directory (universal path)
ROOT_DIR = os.path.dirname(os.path.abspath(__file__))

# Use relative path for open-source release
adata_path = os.path.join(ROOT_DIR, "trajectory_adata.h5ad")
adata = sc.read_h5ad(adata_path)

In [ ]:
# ====== Generate Cell Age Ranking Dataset ======
import pandas as pd
import random
import numpy as np

# Map each age group to corresponding cell indices
age_to_indices = {}
for age, group in adata.obs.groupby('age'):
    # Get position indices (0-based) of cells for current age group
    pos_indices = adata.obs.index.get_indexer(group.index)
    age_to_indices[age] = pos_indices.tolist()

# Check if there are at least 5 distinct ages for sampling
unique_ages = list(age_to_indices.keys())
if len(unique_ages) < 5:
    raise ValueError("Less than 5 distinct ages in dataset, cannot generate required data!")

result_data = []
# Fixed 5 target age groups for sampling
target_ages = [7, 11, 21, 26, 61]

# Generate 5000 synthetic cell ranking samples
for _ in range(5000):
    # Shuffle the 5 target ages randomly
    selected_ages = random.sample(target_ages, len(target_ages))
    
    # Randomly pick one cell index for each selected age
    cell_indices = [random.choice(age_to_indices[age]) for age in selected_ages]

    # Sort ages and map to corresponding cell positions
    sorted_ages = sorted(selected_ages)
    sorted_positions = [selected_ages.index(age) for age in sorted_ages]
    list_str = ','.join([f"cell{i+1}" for i in sorted_positions])

    # Construct one row of data
    row = [
        cell_indices[0], cell_indices[1], cell_indices[2], cell_indices[3], cell_indices[4],
        selected_ages[0], selected_ages[1], selected_ages[2], selected_ages[3], selected_ages[4],
        list_str
    ]
    result_data.append(row)

# Build final DataFrame
columns = [
    'cell1', 'cell2', 'cell3', 'cell4', 'cell5',
    'age1', 'age2', 'age3', 'age4', 'age5',
    'list'
]
cell_result_mask_df = pd.DataFrame(result_data, columns=columns)

In [ ]:
# ====== Model Configuration & Initialization for Cell Tokenization (VQ-VAE) ======
import warnings
warnings.filterwarnings("ignore")

import os
import torch
import argparse

# Argument parser for model training & inference
parser = argparse.ArgumentParser(description="VQ-VAE for Spatial Transcriptomics Cell Tokenization")
parser.add_argument("--train_batch_size", type=int, default=64)
parser.add_argument("--eval_batch_size", type=int, default=2000)
parser.add_argument("--encoder_hidden_dim", type=list, default=[1600,1024,800])
parser.add_argument("--decoder_hidden_dim", type=list, default=[800,1024,1600])
parser.add_argument("--codebook_dim", type=int, default=8)
parser.add_argument("--n", type=int, default=32)
parser.add_argument("--n_codebooks", type=int, default=256)
parser.add_argument("--gamma", type=float, default=5)
parser.add_argument("--beta", type=float, default=10)
parser.add_argument('--weight_decay', type=float, default=5e-4)
parser.add_argument("--lr", type=float, default=1e-4)
parser.add_argument('--max_iter', type=int, default=20)
parser.add_argument('--seed', type=int, default=2024)
parser.add_argument('--gpu', type=int, default=0)
parser.add_argument("--dropout_rate", type=float, default=0.2)
parser.add_argument('--mode', type=str, default="vqvae")
parser.add_argument('--hvg', type=int, default=5000)

args = parser.parse_args([])

# Set computing device (GPU/CPU)
if torch.cuda.is_available():
    args.device = 'cuda'
    torch.cuda.set_device(args.gpu)
else:
    args.device = 'cpu'
print(f"Using device: {args.device}")

# Auto-detect project root & define model path (OPENSOURCE STANDARD)
ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
model_path = os.path.join(ROOT_DIR, "model", "cell_tokenizer", "model.pth")

# Initialize VQ-VAE model
data_dim = 5000
vae = VQVAE(
    data_dim, 
    args.n, 
    args.codebook_dim, 
    args.n_codebooks, 
    args.encoder_hidden_dim, 
    args.decoder_hidden_dim, 
    args.beta, 
    args.gamma, 
    args.dropout_rate
).to(args.device)

# Load pre-trained model weights from relative path
checkpoint = torch.load(model_path, map_location=args.device)
vae.load_state_dict(checkpoint['model'])
vae.eval()
print(f"Pre-trained VQ-VAE model loaded & set to evaluation mode: {model_path}")

In [ ]:
var_names_df = pd.read_csv("/nfs/public/xcx/CellTok/data/5000_gene_names.csv")
gene_names = var_names_df['gene'].tolist()

existing_genes = adata.var_names.isin(gene_names)
adata_filtered = adata[:, existing_genes]

existing_gene_names = adata.var_names[existing_genes].tolist()
result_df = pd.DataFrame(index=adata.obs_names, columns=gene_names)
result_df[:] = 0

if issparse(adata_filtered.X):
    adata_filtered_df = adata_filtered.X.toarray()
else:
    adata_filtered_df = adata_filtered.X

result_df.loc[:, existing_gene_names] = adata_filtered_df

new_adata = sc.AnnData(X=result_df.values.astype(np.float32), obs=adata.obs, var=pd.DataFrame(index=gene_names))
new_adata.var['feature_name'] = gene_names
new_adata.var_names = gene_names

new_adata

In [ ]:
adata = new_adata
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
vae.get_adata_codebook_index(adata, mode=args.mode, device=args.device)

In [ ]:
code_index = adata.obsm["code_index"]
cell_codes = {}
for i in range(code_index.shape[0]):
 cell_codes[i] = code_index[i].tolist() # 将numpy数组转换为列表

In [ ]:
from transformers import (
AutoModelForCausalLM,
AutoTokenizer)

# load the tokenizer of Qwen
tokenizer = AutoTokenizer.from_pretrained("/stor/xcx/nips/single_cell_model_qwen/tokenizer/")

In [ ]:
# # ====== Construct Development Trajectory Cell Sorting Dataset ======
def create_sorted_age_format(row):
 cell_info = [
 ('cell1', row['cell1']),
 ('cell2', row['cell2']),
 ('cell3', row['cell3']),
 ('cell4', row['cell4']),
 ('cell5', row['cell5'])
 ]

 cell_token_parts = []
 for cell_name, cell_idx in cell_info:
    cell_codes_list = cell_codes[cell_idx]
    code_tokens = [f'<code_{code}>' for code in cell_codes_list]
    cell_token_str = f"{cell_name}: <cell_token_start>{''.join(code_tokens)}<cell_token_end>"
    cell_token_parts.append(cell_token_str)

 input_text = ", ".join(cell_token_parts) + \
             ", sort these 5 cells by their age in ascending order and output the cell names (cell1 to cell5) in this order."

 output_text = f"The order is: {row['list']}"

 return {
    'input_text': input_text,
    'output_text': output_text,
    'cell1_idx': row['cell1'],
    'cell2_idx': row['cell2'],
    'cell3_idx': row['cell3'],
    'cell4_idx': row['cell4'],
    'cell5_idx': row['cell5'],
}

# # ====== Construct Development Trajectory Cell Generation Dataset ======
# import pandas as pd

# # Global configuration (adjust according to your dataset)
# INPUT_AGES_SORTED = [4, 12, 21]  # Input ages (sorted in increasing order)
# TARGET_AGE = 31                 # Target age to predict

# # Token format settings
# TARGET_TOKEN = f"{TARGET_AGE}d:"       # For cell token sequence (with colon)
# TARGET_TOKEN_NO_COLON = f"{TARGET_AGE}d"  # For instruction text (no colon)

# def create_sorted_age_format(row):
#     """
#     Construct training data for cell trajectory prediction:
#     1. Sort input cell tokens by age (4d → 12d → 21d)
#     2. Generate instruction text and target output for 31d prediction
#     3. Return formatted Hugging Face sample
#     """
#     # Map cell columns to their indices and ages
#     cell_age_pairs = [
#         ('cell1', row['cell1'], row['age1']),
#         ('cell2', row['cell2'], row['age2']),
#         ('cell3', row['cell3'], row['age3']),
#         ('cell4', row['cell4'], row['age4']),
#     ]

#     input_age_token_dict = {}
#     target_code_tokens = None

#     for cell_name, cell_idx, age in cell_age_pairs:
#         age_int = int(age)

#         # Process input age cells
#         if age_int in INPUT_AGES_SORTED:
#             cell_codes_list = cell_codes.get(cell_idx, [])
#             code_tokens = [f'<code_{code}>' for code in cell_codes_list]
#             day_token = f"{age_int}d:"
#             cell_token_str = f"{day_token}<cell_token_start>{''.join(code_tokens)}<cell_token_end>"
#             input_age_token_dict[age_int] = cell_token_str

#         # Process target age cell (31d)
#         if age_int == TARGET_AGE and target_code_tokens is None:
#             cell_codes_list = cell_codes.get(cell_idx, [])
#             code_tokens = [f'<code_{code}>' for code in cell_codes_list]
#             target_code_tokens = f"<cell_token_start>{''.join(code_tokens)}<cell_token_end>"

#     # Skip invalid samples
#     if target_code_tokens is None or not set(INPUT_AGES_SORTED).issubset(input_age_token_dict.keys()):
#         return None

#     # Sort input tokens by age
#     input_parts = [input_age_token_dict[age] for age in INPUT_AGES_SORTED]

#     # Build instruction text
#     input_ages_no_colon = [f"{age}d" for age in INPUT_AGES_SORTED]
#     input_text = (
#         f"Based on the cell development trajectory of {', '.join(input_ages_no_colon)}, "
#         f"generate the cell code for 31d. "
#         f"Input cell codes: {' '.join(input_parts)}"
#     )

#     # Build target output text
#     output_text = f"{TARGET_TOKEN}{target_code_tokens}"

#     return {
#         'input_text': input_text,
#         'output_text': output_text,
#         'cell1_idx': row['cell1'],
#         'cell2_idx': row['cell2'],
#         'cell3_idx': row['cell3'],
#         'cell4_idx': row['cell4'],
#         'target_cell_age': 31,
#         'input_cell_ages': INPUT_AGES_SORTED
#     }

# # Generate final dataset
# hf_data = []
# for _, row in cell_result_mask_df.iterrows():
#     result = create_sorted_age_format(row)
#     if result is not None:
#         hf_data.append(result)

# hf_df = pd.DataFrame(hf_data)
# print(f"Valid samples generated: {len(hf_df)}")

# if len(hf_df) > 0:
#     print("\nExample input text:\n", hf_df['input_text'].iloc[0])
#     print("\nExample output text:\n", hf_df['output_text'].iloc[0])

In [ ]:
hf_data = []
for _, row in cell_result_mask_df.iterrows():
 hf_data.append(create_sorted_age_format(row))

hf_df = pd.DataFrame(hf_data)
hf_df

In [ ]:
np.random.seed(42)
shuffled_df = hf_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
def preprocess_function_tokenized(examples):
    batch_size = len(examples[text_column])
    targets = [str(x) for x in examples[label_column]]
    model_inputs = tokenizer(examples[text_column])
    labels = tokenizer(targets, add_special_tokens=False)  # don't add bos token because we concatenate with inputs
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i] + [tokenizer.eos_token_id]
        # print(i, sample_input_ids, label_input_ids)
        model_inputs["input_ids"][i] = sample_input_ids + label_input_ids
        labels["input_ids"][i] = [-100] * len(sample_input_ids) + label_input_ids
        model_inputs["attention_mask"][i] = [1] * len(model_inputs["input_ids"][i])
#     print(model_inputs)
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i]
        model_inputs["input_ids"][i] = [tokenizer.pad_token_id] * (
            max_length - len(sample_input_ids)
        ) + sample_input_ids
        model_inputs["attention_mask"][i] = [0] * (max_length - len(sample_input_ids)) + model_inputs[
            "attention_mask"
        ][i]
        labels["input_ids"][i] = [-100] * (max_length - len(sample_input_ids)) + label_input_ids
        model_inputs["input_ids"][i] = model_inputs["input_ids"][i][:max_length]
        model_inputs["attention_mask"][i] = model_inputs["attention_mask"][i][:max_length]
        labels["input_ids"][i] = labels["input_ids"][i][:max_length]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

text_column = "input_text"
label_column = "output_text"
max_length = 250
batch_size = 30

In [ ]:
from datasets import Dataset
hf_df=shuffled_df
hf_dataset = Dataset.from_pandas(hf_df)

In [ ]:
processed_datasets = hf_dataset.map(
preprocess_function_tokenized,
batched=True,
num_proc=10,
load_from_cache_file=True,
desc="Running tokenizer on dataset",
)